# HUMEA strict baseline replay

Two independent 35-epoch DB15K-20 baseline runs. This notebook never starts full training. Strict determinism failures stop immediately and preserve available evidence.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

MODE = "replay"         # replay | validate
EXPERIMENT = "db15k-20"
BATCH_SIZE = 512          # Paper setting; checkpointed for Kaggle T4
DOWNLOAD_DATA_IF_MISSING = True
WANDB_MODE = "online"   # requires the enabled WANDB_API_KEY Kaggle secret
WANDB_PROJECT = "humea-reproduction"
WANDB_ENTITY = "diencongle"
WORK_ROOT = Path("/kaggle/working/HUMEA")
INPUT_ROOT = Path("/kaggle/input")
DATA_URL = "https://drive.google.com/file/d/1a3aou1qe7Yzq6y_khyTq1La_kf3UI7sC/view?usp=drive_link"
SOURCE_REVISION = "feat/strict-replay"  # Resolve and print the source commit; deployment pins this SHA
CAMPAIGN = "db15k20-strict-replay-20261023"
assert MODE in {"validate", "replay"}
print("Mode:", MODE, "Experiment:", EXPERIMENT, "Batch size:", BATCH_SIZE)

In [ ]:
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
subprocess.run(["nvidia-smi"], check=False)

## Load pinned public reproduction source

The source comes from the public GitHub fork. Existing attached Kaggle data is reused; API keys stay in Kaggle Secrets.


In [ ]:
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
subprocess.run(["git", "clone", "--quiet", "https://github.com/lecongdien2000/HUMEA.git", str(WORK_ROOT)], check=True)
subprocess.run(["git", "checkout", "--quiet", SOURCE_REVISION], cwd=WORK_ROOT, check=True)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORK_ROOT, text=True).strip()
print("Source commit:", actual_revision)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["WANDB_RUN_GROUP"] = CAMPAIGN
os.environ["WANDB_TAGS"] = "db15k20,strict-replay,diagnostic,seed42"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(WORK_ROOT / "requirements-kaggle.txt")], check=True)
# Never store an API key in the notebook or print it.
assert WANDB_MODE in {"online", "offline", "disabled"}
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
if WANDB_MODE == "online":
    if not WANDB_ENTITY:
        raise ValueError("Set WANDB_ENTITY to your W&B username/team")
    if not os.environ.get("WANDB_API_KEY"):
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    if not os.environ.get("WANDB_API_KEY"):
        raise ValueError("Enable the WANDB_API_KEY Kaggle secret before running")
print("W&B mode:", WANDB_MODE, "project:", WANDB_PROJECT)

import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
print("W&B credential loaded; online run initialization is checked by the smoke jobs.")


## Locate or download the processed features

Attached, already-extracted `mmkb-datasets` is preferred. Otherwise the notebook extracts an attached `data.zip`; with Internet enabled it can download the authors' archive once.

In [ ]:
expected = WORK_ROOT / "data" / "mmkb-datasets"
if not expected.is_dir():
    extracted_candidates = [p for p in INPUT_ROOT.rglob("mmkb-datasets") if p.is_dir()]
    if extracted_candidates:
        data_link = WORK_ROOT / "data"
        if data_link.exists() or data_link.is_symlink():
            if data_link.is_dir() and not data_link.is_symlink():
                shutil.rmtree(data_link)
            else:
                data_link.unlink()
        data_link.symlink_to(extracted_candidates[0].parent, target_is_directory=True)
        print("Using attached extracted data:", extracted_candidates[0])
    else:
        archives = list(INPUT_ROOT.rglob("data.zip"))
        archive = archives[0] if archives else Path("/kaggle/working/humea-data.zip")
        if not archives:
            if not DOWNLOAD_DATA_IF_MISSING:
                raise FileNotFoundError("No attached data.zip and downloading is disabled")
            subprocess.run(["gdown", "--fuzzy", DATA_URL, "-O", str(archive)], check=True)
        (WORK_ROOT / "data").mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as data_zip:
            data_zip.extractall(WORK_ROOT / "data")
        print("Extracted", archive)
assert expected.is_dir(), f"Expected dataset directory was not created: {expected}"
print("Dataset root:", expected)

In [ ]:
base_command = [
    sys.executable, "kaggle_runner.py", "--repo-root", str(WORK_ROOT),
    "--batch-size", str(BATCH_SIZE),
]
validate = subprocess.run(base_command + ["validate"], cwd=WORK_ROOT)
if validate.returncode != 0:
    raise RuntimeError("Preflight validation failed; review the missing-file output above")

In [ ]:
import json, signal
artifacts = WORK_ROOT / "artifacts" / "strict-replay"
command = [sys.executable, "replay_runner.py", "--epochs", "35",
           "--wandb-mode", WANDB_MODE, "--wandb-entity", WANDB_ENTITY,
           "--wandb-project", WANDB_PROJECT, "--artifacts-dir", str(artifacts)]
console_log = Path("/kaggle/working/HUMEA-strict-replay.log")
try:
    if MODE == "replay":
        with console_log.open("w") as stream:
            process = subprocess.Popen(command, cwd=WORK_ROOT, stdout=stream,
                stderr=subprocess.STDOUT, start_new_session=True)
            try:
                returncode = process.wait(timeout=45 * 60)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
                raise RuntimeError("45-minute replay budget reached; partial evidence preserved")
        print(console_log.read_text()[-8000:])
        comparison = artifacts / "replay-comparison.json"
        if comparison.exists():
            print(comparison.read_text())
        if returncode:
            raise RuntimeError("Strict replay did not pass; inspect saved log and comparison before any full run")
finally:
    if (WORK_ROOT / "artifacts").is_dir():
        archive = shutil.make_archive("/kaggle/working/HUMEA-strict-replay-results", "zip",
                                      WORK_ROOT, "artifacts")
        print("Evidence archive:", archive)
